<a href="https://colab.research.google.com/github/Mario-Cattaneo/SemesterProject/blob/main/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Clone the repository
!git clone https://github.com/ricsamikwa/DiSNet.git
%cd DiSNet

# Install any required packages (check requirements.txt if present in the repo)
# !pip install -r requirements.txt

Cloning into 'DiSNet'...
remote: Enumerating objects: 579, done.
remote: Counting objects: 100% (579/579), done.
remote: Compressing objects: 100% (354/354), done.
remote: Total 579 (delta 346), reused 447 (delta 220), pack-reused 0 (from 0)
Receiving objects: 100% (579/579), 1.61 MiB | 7.58 MiB/s, done.
Resolving deltas: 100% (346/346), done.
/content/DiSNet


In [2]:
# 1. Patch the model_convert.py file programmatically
import torchvision.models as models

with open('main/model_convert.py', 'r') as file:
    content = file.read()

# Replace the outdated torch.hub load with direct torchvision import
old_line = "Vgg_pretrined = torch.hub.load('pytorch/vision:v0.9.0', 'vgg16_bn', pretrained=True)"
new_line = "import torchvision.models as models\nVgg_pretrined = models.vgg16_bn(pretrained=True)"
content = content.replace(old_line, new_line)

# Replace the hardcoded author path with the Colab root path
old_path = "/home/eric/.cache/torch/hub/checkpoints/vgg16_bn-6c64b313.pth"
new_path = "/root/.cache/torch/hub/checkpoints/vgg16_bn-6c64b313.pth"
content = content.replace(old_path, new_path)

with open('main/model_convert.py', 'w') as file:
    file.write(content)

print("Successfully patched main/model_convert.py")

# 2. Run the conversion script to download and modify the weights
!mkdir -p main/models
!python3 main/model_convert.py

Successfully patched main/model_convert.py
conv_layers.0.0.weight : torch.Size([64, 3, 3, 3])
conv_layers.0.0.bias : torch.Size([64])
conv_layers.0.1.weight : torch.Size([64])
conv_layers.0.1.bias : torch.Size([64])
conv_layers.1.0.weight : torch.Size([64, 64, 3, 3])
conv_layers.1.0.bias : torch.Size([64])
conv_layers.1.1.weight : torch.Size([64])
conv_layers.1.1.bias : torch.Size([64])
conv_layers.3.0.weight : torch.Size([128, 64, 3, 3])
conv_layers.3.0.bias : torch.Size([128])
conv_layers.3.1.weight : torch.Size([128])
conv_layers.3.1.bias : torch.Size([128])
conv_layers.4.0.weight : torch.Size([128, 128, 3, 3])
conv_layers.4.0.bias : torch.Size([128])
conv_layers.4.1.weight : torch.Size([128])
conv_layers.4.1.bias : torch.Size([128])
conv_layers.6.0.weight : torch.Size([256, 128, 3, 3])
conv_layers.6.0.bias : torch.Size([256])
conv_layers.6.1.weight : torch.Size([256])
conv_layers.6.1.bias : torch.Size([256])
conv_layers.7.0.weight : torch.Size([256, 256, 3, 3])
conv_layers.7.0.bias

In [3]:
%%writefile configurations.py
import numpy as np

SERVER_ADDR = '192.168.1.38'
SERVER_PORT = 43000

N = 50000 # Data length

model_cfg = {
    'VGG' : [
        ('C', 3, 32, 3, 32*32*32, 32*32*32*3*3*3), ('M', 32, 32, 2, 32*16*16, 0),
        ('C', 32, 64, 3, 64*16*16, 64*16*16*3*3*32), ('M', 64, 64, 2, 64*8*8, 0),
        ('C', 64, 64, 3, 64*8*8, 64*8*8*3*3*64),
        ('D', 8*8*64, 128, 1, 64, 128*8*8*64),
        ('C', 64, 64, 3, 64*8*8, 64*8*8*3*3*64),
        ('D', 8*8*64, 128, 1, 64, 128*8*8*64),
        ('D', 128, 10, 1, 10, 128*10)
    ]
}

model_name = 'VGG'
model_size = 1.28
split_layer = [2, 3, 2]
model_len = 7
DEVICE_PACE_RATE = 10
rate = 10

max_par_partitions = [6, 5, 4, 3]

trans_powers = [1, 0.8, 1.1]
comp_powers = [3.8, 2.2, 1.2]
device_power_groups = [3, 7, 10]

pos_max_par_partitions = [4, 9, 12, 18]
layer_range = np.array([[0, 4], [4, 9], [9, 14], [14, 18]])

R = 100
LR = 0.01
B = 100

K = 10

iteration = {'192.168.1.33': 5, '192.168.1.41': 5, '192.168.1.40': 10, '192.168.1.42': 5, '192.168.1.43': 5}
random = True
random_seed = 0

Overwriting configurations.py


In [4]:
import sys
import os
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
import importlib # Add this line

# Ensure both root and main directories are in the Python path
sys.path.append(os.getcwd())
sys.path.append(os.path.join(os.getcwd(), 'main'))

from models.model_vgg16 import VGG16
# Explicitly reload the module to ensure the patched version (ceil_mode=True) is used
if 'models.model_vgg16' in sys.modules:
    importlib.reload(sys.modules['models.model_vgg16'])

from inference import *
from utils import *
from network import *
import configurations

def run_simulation(energy_sensitivity=0.0, max_bandwidth=10.0, mesh_network_id=0):
    """
    Runs the DiSNet simulation and returns (latency, energy, top5_accuracy).
    Set energy_sensitivity = 0.0 to optimize purely for latency.
    """
    # Load and preprocess sample image
    filename = "input/dog.jpg"
    input_image = Image.open(filename)
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    input_batch = preprocess(input_image).unsqueeze(0)

    # Load model
    model = VGG16()
    model_dict = model.state_dict()
    model_dict.update(torch.load("main/vgg16-modify.pth"))
    model.load_state_dict(model_dict)
    model.eval()

    device_pace_rate = configurations.DEVICE_PACE_RATE

    if torch.cuda.is_available():
        input_batch = input_batch.to('cuda:0')
        model.to('cuda:0')

    # Load network graph
    G = read_graph(mesh_network_id)
    max_par_partitions = configurations.max_par_partitions

    # Fixed input/output nodes for consistency
    input_node = 8
    output_node = 4

    # Select path based on energy sensitivity
    selected_path = select_path(G, input_node, output_node, energy_sensitivity)

    current_point_on_path = 0
    split_ratio = []
    devices = []
    trans_rate_forward = []
    par_trans_rate = []
    execution_path = []

    for i in range(len(max_par_partitions)):
        current_max_par_partitions = max_par_partitions[i]
        p = determine_opt_neighbours(G, selected_path, current_max_par_partitions, current_point_on_path, energy_sensitivity)
        execution_path.append(p)
        current_point_on_path = selected_path.index(p)

        current_neighbours = []
        for n in G.neighbors(p):
            current_neighbours.append([n, G.nodes[n]['weight'], G[p][n]['weight']])

        if current_point_on_path >= len(selected_path)-1:
            path_point = current_point_on_path-1
            next_device = selected_path[path_point]
            current_neighbours.append([p, G.nodes[p]['weight'], G[p][next_device]['weight']])
            trans_rate_forward.append(G[p][selected_path[current_point_on_path-1]]['weight'])
        else:
            path_point = current_point_on_path+1
            next_device = selected_path[path_point]
            current_neighbours.append([p, G.nodes[p]['weight'], G[p][next_device]['weight']])
            trans_rate_forward.append(G[p][selected_path[current_point_on_path+1]]['weight'])

        if len(current_neighbours) < current_max_par_partitions:
            current_max_par_partitions = len(current_neighbours)
            horizontal_split_ratio, device, in_throughput = find_split_ratio(current_neighbours)
            split_ratio.append(horizontal_split_ratio)
            par_trans_rate.append(in_throughput)
            devices.append(device)
        else:
            subset_current_neighbours = select_subset(current_neighbours, current_max_par_partitions)
            horizontal_split_ratio, device, in_throughput = find_split_ratio(subset_current_neighbours)
            split_ratio.append(horizontal_split_ratio)
            par_trans_rate.append(in_throughput)
            devices.append(device)

    trans_rate_forward = cap_trans_rate(max_bandwidth, trans_rate_forward)
    par_trans_rate = cap_trans_rate(max_bandwidth, par_trans_rate)
    layer_range = configurations.layer_range
    partition_input = create_partition_input(split_ratio)
    comp_rate = split_ratio

    output = input_batch
    infer_time = []
    trans_time_seq = []
    infer_energy = 0

    with torch.no_grad():
        for j in range(len(max_par_partitions)):
            output, sub_infer_time = opt_DiSNet(output, layer_range[j], partition_input[layer_range[j,0]:layer_range[j,1]], par_trans_rate[j], comp_rate[j], split_ratio[j], model)

            fowrd_trans_time = 0
            if j == len(max_par_partitions)-1:
                fowrd_trans_time = 0
            else:
                if execution_path[j] == execution_path[j +1]:
                    if devices[j] == devices[j + 1]:
                        fowrd_trans_time = 0
                    else:
                        fowrd_trans_time = trans_time_forward(output, trans_rate_forward[j]*10, layer_range[j])
                else:
                    fowrd_trans_time = trans_time_forward(output, trans_rate_forward[j], layer_range[j])

            fowrd_trans_time = fowrd_trans_time / device_pace_rate
            infer_energy += partition_energy(devices[j], sub_infer_time, fowrd_trans_time)
            trans_time_seq.append(fowrd_trans_time)
            infer_time.append(sub_infer_time)

        total_latency = np.sum(infer_time) + np.sum(trans_time_seq)
        probabilities = torch.nn.functional.softmax(output[0], dim=0)
        top5_prob, _ = torch.topk(probabilities, 5)
        top5_accuracy = sum(top5_prob).item()

    return total_latency, infer_energy, top5_accuracy

In [5]:
# Run a test simulation with 10 Mbps bandwidth limit
latency, energy, accuracy = run_simulation(energy_sensitivity=0.0, max_bandwidth=10.0)

print("--- Simulation Results ---")
print(f"End-to-End Latency: {latency:.4f} seconds")
print(f"Energy Consumption: {energy:.4f} Joules")
print(f"Top-5 Accuracy:     {accuracy:.4f}")

TypeError: super(type, obj): obj must be an instance or subtype of type

In [ ]:
import torch

class BaseCodingScheme:
    """
    Base class for all activation coding/compression schemes.
    """
    def __init__(self, name, bits=32):
        self.name = name
        self.bits = bits  # Effective bits per element after compression

    def compress(self, tensor):
        raise NotImplementedError("Subclasses must implement the compress method.")


class NoCompression(BaseCodingScheme):
    """
    Baseline: Transmits raw 32-bit floating-point activations.
    """
    def __init__(self):
        super().__init__(name="Uncompressed (32-bit)", bits=32)

    def compress(self, tensor):
        return tensor


class UniformQuantization(BaseCodingScheme):
    """
    Quantizes activations uniformly to a specified bit-width.
    """
    def __init__(self, bits=8):
        super().__init__(name=f"Quantization ({bits}-bit)", bits=bits)

    def compress(self, tensor):
        if self.bits >= 32:
            return tensor

        min_val = tensor.min()
        max_val = tensor.max()
        scale = (max_val - min_val) / (2**self.bits - 1)

        if scale == 0:
            return tensor

        quantized = torch.round((tensor - min_val) / scale)
        dequantized = quantized * scale + min_val
        return dequantized


class MagnitudePruning(BaseCodingScheme):
    """
    Keeps only the top fraction of activations by magnitude, setting the rest to zero.
    Assumes sparse encoding is used to scale the transmitted bits.
    """
    def __init__(self, keep_ratio=0.5):
        super().__init__(name=f"Pruning ({int(keep_ratio*100)}% kept)", bits=32 * keep_ratio)
        self.keep_ratio = keep_ratio

    def compress(self, tensor):
        if self.keep_ratio >= 1.0:
            return tensor

        # Flatten tensor to find the threshold
        flat_tensor = tensor.abs().view(-1)
        k = int(self.keep_ratio * flat_tensor.numel())

        if k == 0:
            return torch.zeros_like(tensor)

        threshold, _ = torch.topk(flat_tensor, k)
        thresh_val = threshold[-1]

        # Apply hard thresholding
        mask = tensor.abs() >= thresh_val
        return tensor * mask

In [ ]:
import sys
import os
import torch
import numpy as np
from PIL import Image
from torchvision import transforms

# Ensure both root and main directories are in the Python path
sys.path.append(os.getcwd())
sys.path.append(os.path.join(os.getcwd(), 'main'))

from models.model_vgg16 import VGG16
from inference import *
from utils import *
from network import *
import configurations

def run_coded_simulation(model, coding_scheme, energy_sensitivity=0.0, max_bandwidth=10.0, mesh_network_id=0):
    """
    Runs the DiSNet simulation using a specific coding scheme and a pre-loaded model.
    """
    filename = "input/dog.jpg"
    input_image = Image.open(filename)
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    input_batch = preprocess(input_image).unsqueeze(0)

    device_pace_rate = configurations.DEVICE_PACE_RATE

    if torch.cuda.is_available():
        input_batch = input_batch.to('cuda:0')
        model.to('cuda:0')

    G = read_graph(mesh_network_id)
    max_par_partitions = configurations.max_par_partitions
    input_node = 8
    output_node = 4

    selected_path = select_path(G, input_node, output_node, energy_sensitivity)
    current_point_on_path = 0
    split_ratio = []
    devices = []
    trans_rate_forward = []
    par_trans_rate = []
    execution_path = []

    for i in range(len(max_par_partitions)):
        current_max_par_partitions = max_par_partitions[i]
        p = determine_opt_neighbours(G, selected_path, current_max_par_partitions, current_point_on_path, energy_sensitivity)
        execution_path.append(p)
        current_point_on_path = selected_path.index(p)

        current_neighbours = []
        for n in G.neighbors(p):
            current_neighbours.append([n, G.nodes[n]['weight'], G[p][n]['weight']])

        if current_point_on_path >= len(selected_path)-1:
            path_point = current_point_on_path-1
            next_device = selected_path[path_point]
            current_neighbours.append([p, G.nodes[p]['weight'], G[p][next_device]['weight']])
            trans_rate_forward.append(G[p][selected_path[current_point_on_path-1]]['weight'])
        else:
            path_point = current_point_on_path+1
            next_device = selected_path[path_point]
            current_neighbours.append([p, G.nodes[p]['weight'], G[p][next_device]['weight']])
            trans_rate_forward.append(G[p][selected_path[current_point_on_path+1]]['weight'])

        if len(current_neighbours) < current_max_par_partitions:
            current_max_par_partitions = len(current_neighbours)
            horizontal_split_ratio, device, in_throughput = find_split_ratio(current_neighbours)
            split_ratio.append(horizontal_split_ratio)
            par_trans_rate.append(in_throughput)
            devices.append(device)
        else:
            subset_current_neighbours = select_subset(current_neighbours, current_max_par_partitions)
            horizontal_split_ratio, device, in_throughput = find_split_ratio(subset_current_neighbours)
            split_ratio.append(horizontal_split_ratio)
            par_trans_rate.append(in_throughput)
            devices.append(device)

    trans_rate_forward = cap_trans_rate(max_bandwidth, trans_rate_forward)
    par_trans_rate = cap_trans_rate(max_bandwidth, par_trans_rate)
    layer_range = configurations.layer_range
    partition_input = create_partition_input(split_ratio)
    comp_rate = split_ratio

    output = input_batch
    infer_time = []
    trans_time_seq = []
    infer_energy = 0

    with torch.no_grad():
        for j in range(len(max_par_partitions)):
            output, sub_infer_time = opt_DiSNet(output, layer_range[j], partition_input[layer_range[j,0]:layer_range[j,1]], par_trans_rate[j], comp_rate[j], split_ratio[j], model)

            # Apply the selected coding scheme to the activations at the split boundary
            output = coding_scheme.compress(output)

            fowrd_trans_time = 0
            if j == len(max_par_partitions)-1:
                fowrd_trans_time = 0
            else:
                if execution_path[j] == execution_path[j +1]:
                    if devices[j] == devices[j + 1]:
                        fowrd_trans_time = 0
                    else:
                        fowrd_trans_time = trans_time_forward(output, trans_rate_forward[j]*10, layer_range[j])
                else:
                    fowrd_trans_time = trans_time_forward(output, trans_rate_forward[j], layer_range[j])

            # Scale transmission latency by the bit-width ratio (e.g., 8-bit vs 32-bit)
            bit_ratio = coding_scheme.bits / 32.0
            fowrd_trans_time = (fowrd_trans_time * bit_ratio) / device_pace_rate

            infer_energy += partition_energy(devices[j], sub_infer_time, fowrd_trans_time)
            trans_time_seq.append(fowrd_trans_time)
            infer_time.append(sub_infer_time)

        total_latency = np.sum(infer_time) + np.sum(trans_time_seq)
        probabilities = torch.nn.functional.softmax(output[0], dim=0)
        top5_prob, _ = torch.topk(probabilities, 5)
        top5_accuracy = sum(top5_prob).item()

    return total_latency, infer_energy, top5_accuracy


def evaluate_compression_tradeoff(model, coding_schemes, fixed_bandwidth=10.0, energy_sensitivity=0.0, curve_label="Curve"):
    """
    Evaluates a list of coding schemes under a SINGLE fixed bandwidth.
    """
    importlib.reload(configurations)

    latencies = []
    energies = []
    accuracies = []

    print(f"Evaluating curve: {curve_label} (Fixed Bandwidth: {fixed_bandwidth} Mbps)")
    for scheme in coding_schemes:
        try:
            latency, energy, accuracy = run_coded_simulation(
                model=model,
                coding_scheme=scheme,
                energy_sensitivity=energy_sensitivity,
                max_bandwidth=fixed_bandwidth
            )
            latencies.append(latency)
            energies.append(energy)
            accuracies.append(accuracy)
            print(f"  Scheme: {scheme.name:25s} | Latency: {latency:.4f}s | Accuracy: {accuracy:.4f}")
        except Exception as e:
            print(f"  Failed for scheme {scheme.name}: {e}")

    return {
        "label": curve_label,
        "latencies": latencies,
        "energies": energies,
        "accuracies": accuracies,
        "schemes": [s.name for s in coding_schemes]
    }

In [ ]:
import torch
import torch.nn as nn

class BottleneckNet(nn.Module):
    def __init__(self, in_channels, bottleneck_channels, bits, is_conv=True):
        super().__init__()
        self.is_conv = is_conv
        self.bits = bits

        if is_conv:
            self.encoder = nn.Conv2d(in_channels, bottleneck_channels, kernel_size=1, bias=False)
            self.decoder = nn.Conv2d(bottleneck_channels, in_channels, kernel_size=1, bias=False)
        else:
            self.encoder = nn.Linear(in_features=in_channels, out_features=bottleneck_channels, bias=False)
            self.decoder = nn.Linear(in_features=bottleneck_channels, out_features=in_channels, bias=False)

    def quantize(self, x):
        if self.bits >= 32:
            return x
        min_val = x.min()
        max_val = x.max()
        scale = (max_val - min_val) / (2**self.bits - 1)
        if scale == 0:
            return x
        # Straight-Through Estimator (STE)
        quantized = torch.round((x - min_val) / scale)
        dequantized = quantized * scale + min_val
        return x + (dequantized - x).detach()

    def forward(self, x):
        latent = self.encoder(x)
        latent_q = self.quantize(latent)
        reconstructed = self.decoder(latent_q)
        return reconstructed


class DAG_VDDIB(BaseCodingScheme):
    """
    Our Proposed Strategy: Variational Distributed Deterministic Information Bottleneck.
    A trainable channel-reduction bottleneck.
    """
    def __init__(self, bottleneck_channels, bits=4):
        super().__init__(name=f"DAG-VDDIB (C'={bottleneck_channels}, {bits}-bit)", bits=bits)
        self.bottleneck_channels = bottleneck_channels
        self.initial_bits = bits
        self.nets = nn.ModuleDict()

    def compress(self, tensor):
        is_conv = (tensor.dim() == 4)
        in_features = tensor.shape[1]
        key = f"{in_features}_{'conv' if is_conv else 'linear'}"

        if key not in self.nets:
            b_features = self.bottleneck_channels
            if b_features >= in_features:
                b_features = max(1, in_features // 2)

            net = BottleneckNet(in_features, b_features, self.initial_bits, is_conv=is_conv)
            net = net.to(tensor.device)
            self.nets[key] = net

            # Update effective bits
            total_effective_bits = 0
            for k, n in self.nets.items():
                parts = k.split('_')
                c_in = int(parts[0])
                c_out = n.encoder.out_channels if n.is_conv else n.encoder.out_features
                total_effective_bits += self.initial_bits * (c_out / c_in)
            self.bits = total_effective_bits / len(self.nets)

        return self.nets[key](tensor)

In [ ]:
import torch.optim as optim

def train_vddib_bottleneck(coding_scheme, model, partition_input, par_trans_rate, comp_rate, split_ratio, num_steps=100, lr=0.01):
    """
    Trains the bottleneck layers of the DAG-VDDIB scheme to reconstruct
    the activations of the frozen VGG-16 model using the actual simulation path.
    """
    print(f"\n--- Training Bottleneck for {coding_scheme.name} ---")

    # 1. Freeze the VGG-16 model parameters
    for param in model.parameters():
        param.requires_grad = False
    model.eval()

    # 2. Initialize the bottleneck networks by running a single forward pass
    x_input = torch.randn(1, 3, 224, 224)
    if torch.cuda.is_available():
        x_input = x_input.to('cuda:0')
        model.to('cuda:0')

    output = x_input
    layer_range = configurations.layer_range

    # Run a forward pass to trigger the dynamic initialization of self.nets
    with torch.no_grad():
        for j in range(len(layer_range)):
            output, _ = opt_DiSNet(output, layer_range[j], partition_input[layer_range[j,0]:layer_range[j,1]], par_trans_rate[j], comp_rate[j], split_ratio[j], model)
            output = coding_scheme.compress(output) # This initializes the bottleneck layers

    # 3. Set up the optimizer for the bottleneck parameters only
    optimizer = optim.Adam(coding_scheme.nets.parameters(), lr=lr)
    criterion = nn.MSELoss()

    # 4. Fine-tune the bottleneck weights using reconstruction loss (Batch Size = 1)
    for step in range(num_steps):
        optimizer.zero_grad()
        loss = 0

        # Generate a representative activation (Must be batch size 1 to align with DiSNet)
        x_batch = torch.randn(1, 3, 224, 224)
        if torch.cuda.is_available():
            x_batch = x_batch.to('cuda:0')

        output = x_batch
        for j in range(len(layer_range)):
            with torch.no_grad():
                output, _ = opt_DiSNet(output, layer_range[j], partition_input[layer_range[j,0]:layer_range[j,1]], par_trans_rate[j], comp_rate[j], split_ratio[j], model)

            # We want the bottleneck to reconstruct the output perfectly
            reconstructed = coding_scheme.compress(output)
            loss += criterion(reconstructed, output)
            output = reconstructed

        loss.backward()
        optimizer.step()

        if (step + 1) % 20 == 0:
            print(f"  Step {step+1}/{num_steps} | Reconstruction Loss: {loss.item():.6f}")

    print("--- Training Complete ---")

In [ ]:
# 1. Load the model once globally
global_model = VGG16()
global_model_dict = global_model.state_dict()
global_model_dict.update(torch.load("main/vgg16-modify.pth"))
global_model.load_state_dict(global_model_dict)
global_model.eval()
if torch.cuda.is_available():
    global_model = global_model.to('cuda:0')

# 2. Define the constant network bandwidth for the evaluation
fixed_bandwidth = 5.0  # Mbps

# 3. Define the Quantization curve
quantization_curve = [
    UniformQuantization(bits=2),
    UniformQuantization(bits=4),
    UniformQuantization(bits=8),
    NoCompression()
]

# 4. Define the Pruning curve
pruning_curve = [
    MagnitudePruning(keep_ratio=0.1),
    MagnitudePruning(keep_ratio=0.25),
    MagnitudePruning(keep_ratio=0.5),
    NoCompression()
]

# 5. Define our Proposed DAG-VDDIB curve
vddib_curve = [
    DAG_VDDIB(bottleneck_channels=16, bits=2),  # Extreme compression
    DAG_VDDIB(bottleneck_channels=32, bits=4),  # Medium compression
    DAG_VDDIB(bottleneck_channels=64, bits=8),  # Light compression
    NoCompression()
]

# 6. Extract the actual simulation parameters from the graph for training
G = read_graph(0)
selected_path = select_path(G, 8, 4, 0.0)
current_point_on_path = 0
split_ratio = []
devices = []
trans_rate_forward = []
par_trans_rate = []

for i in range(len(configurations.max_par_partitions)):
    current_max_par_partitions = configurations.max_par_partitions[i]
    p = determine_opt_neighbours(G, selected_path, current_max_par_partitions, current_point_on_path, 0.0)
    current_point_on_path = selected_path.index(p)
    current_neighbours = []
    for n in G.neighbors(p):
        current_neighbours.append([n, G.nodes[n]['weight'], G[p][n]['weight']])
    if current_point_on_path >= len(selected_path)-1:
        current_neighbours.append([p, G.nodes[p]['weight'], G[p][selected_path[current_point_on_path-1]]['weight']])
    else:
        current_neighbours.append([p, G.nodes[p]['weight'], G[p][selected_path[current_point_on_path+1]]['weight']])

    horizontal_split_ratio, device, in_throughput = find_split_ratio(current_neighbours)
    split_ratio.append(horizontal_split_ratio)
    par_trans_rate.append(in_throughput)

par_trans_rate = cap_trans_rate(fixed_bandwidth, par_trans_rate)
partition_input = create_partition_input(split_ratio)
comp_rate = split_ratio

# 7. Train the bottlenecks of our proposed schemes using the global model
for scheme in vddib_curve:
    if isinstance(scheme, DAG_VDDIB):
        train_vddib_bottleneck(
            coding_scheme=scheme,
            model=global_model,
            partition_input=partition_input,
            par_trans_rate=par_trans_rate,
            comp_rate=comp_rate,
            split_ratio=split_ratio,
            num_steps=40,
            lr=0.01
        )

print("=" * 60)

# 8. Run evaluations using the global model
results_quant = evaluate_compression_tradeoff(
    model=global_model,
    coding_schemes=quantization_curve,
    fixed_bandwidth=fixed_bandwidth,
    curve_label="Uniform Quantization"
)
print("-" * 60)

results_pruning = evaluate_compression_tradeoff(
    model=global_model,
    coding_schemes=pruning_curve,
    fixed_bandwidth=fixed_bandwidth,
    curve_label="Magnitude Pruning"
)
print("-" * 60)

results_vddib = evaluate_compression_tradeoff(
    model=global_model,
    coding_schemes=vddib_curve,
    fixed_bandwidth=fixed_bandwidth,
    curve_label="DAG-VDDIB (Proposed)"
)
print("-" * 60)

# 9. Visualize and compare all three curves
plot_compression_tradeoffs([results_quant, results_pruning, results_vddib], fixed_bandwidth)


--- Training Bottleneck for DAG-VDDIB (C'=16, 2-bit) ---


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x7168 and 25088x4096)